In [1]:
import sys
print(sys.executable)

import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

h:\4. Projects_envs\8. Image caption CNN+LSTM\env\Scripts\python.exe
CUDA: True
GPU: NVIDIA GeForce GTX 1650 Ti


In [2]:
import torch

print("Torch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)
print("GPU Name:", torch.cuda.get_device_name(0))

Torch Version: 2.5.1+cu121
CUDA Available: True
CUDA Version: 12.1
GPU Name: NVIDIA GeForce GTX 1650 Ti


In [12]:
import os
import pickle
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from tqdm import tqdm
import re
from collections import Counter
import random

## 2. DataPaths

In [13]:
IMAGE_FOLDER = r"H:\4. Projects_envs\8. Image caption CNN+LSTM\fliker8k\Images"

CAPTION_FILE = r"H:\4. Projects_envs\8. Image caption CNN+LSTM\fliker8k\captions.txt"

OUTPUT_FILE = r"H:\4. Projects_envs\8. Image caption CNN+LSTM\Images_features.pkl"

VOCAB_FILE = r"H:\4. Projects_envs\8. Image caption CNN+LSTM\Caption_vocab.pkl"

NUMERICAL_CAPTIONS_FILE = r"H:\4. Projects_envs\8. Image caption CNN+LSTM\Numerical_captions.pkl"

## 3. Load Pretrained model - (resnet-50)


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [15]:
resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
modules = list(resnet.children())[:-1]

model = nn.Sequential(*modules)
model = model.to(device)
model.eval()

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


#### 3.1 Image Transformation

In [16]:
import torch.nn as nn

# Load pretrained model
resnet = models.resnet50(pretrained=True)

# Remove last fully connected layer
resnet.fc = nn.Identity()

# Move to GPU
resnet = resnet.to(device)

# Evaluation mode
resnet.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

#### 3.2 Image Feature Extraction

In [18]:
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),   # ResNet expected size
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],   # ImageNet mean
        std=[0.229, 0.224, 0.225]     # ImageNet std
    )
])

In [ ]:
features_dict = {}

image_files = os.listdir(IMAGE_FOLDER)

with torch.no_grad():
    for image_name in tqdm(image_files):

        image_path = os.path.join(IMAGE_FOLDER, image_name)

        image = Image.open(image_path).convert("RGB")
        image = transform(image).unsqueeze(0).to(device)

        feature = resnet(image)
        feature = feature.squeeze(0)
        feature = feature.cpu().numpy()

        features_dict[image_name] = feature

#### 3.3 Saving as a pkl file

In [ ]:
import os

print("Files one level up:")
print(os.listdir(".."))

Files one level up:
['Caption_vocab.pkl', 'env', 'fliker8k', 'Numerical_captions.pkl']


In [ ]:

# ------------------------------
# Save as Pickle
# ------------------------------



with open(r"H:\4. Projects_envs\8. Image caption CNN+LSTM\Images_features.pkl", "wb") as f:
    pickle.dump(features_dict, f)

print("Saved successfully!")
print("Total images:", len(features_dict))



print("Features saved to:", OUTPUT_FILE)
print("Total images processed:", len(features_dict))

Saved successfully!
Total images: 8091
Features saved to: H:\4. Projects_envs\8. Image caption CNN+LSTM\Images_features.pkl
Total images processed: 8091


## 4 Caption data cleaning and creating vocab

#### 4.1 cleaing text

In [20]:
# Clean Text 
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    text = text.strip()
    return text

#### 4.2 read caption

In [24]:
# -----------------------------
# 4.2 Read Captions (FIXED)
# -----------------------------

captions_dict = {}
all_words = []

with open(CAPTION_FILE, "r", encoding="utf-8") as f:
    lines = f.readlines()

for line in lines[1:]:   # Skip header (image,caption)
    parts = line.strip().split(",", 1)

    if len(parts) != 2:
        continue

    image_id = parts[0]
    caption = parts[1]

    caption = clean_text(caption)
    words = caption.split()

    all_words.extend(words)

    if image_id not in captions_dict:
        captions_dict[image_id] = []

    captions_dict[image_id].append(words)

print("Total images with captions:", len(captions_dict))

Total images with captions: 8091


#### 4.3 creating vocb

In [41]:
MIN_WORD_FREQ = 5
word_counts = Counter(all_words)

vocab = {
    "<pad>": 0,
    "<start>": 1,
    "<end>": 2,
    "<unk>": 3
}

index = 4

for word, count in word_counts.items():
    if count >= MIN_WORD_FREQ:
        vocab[word] = index
        index += 1
vocab_size = len(vocab)   # 🔥 IMPORTANT FIX
print("Vocabulary size:", vocab_size)

Vocabulary size: 2988


#### 4.4 capton flating

In [27]:
# Convert Captions to Numbers
numerical_captions = {}

for image_id, captions in captions_dict.items():
    numerical_captions[image_id] = []

    for caption in captions:
        encoded = [vocab["<start>"]]

        for word in caption:
            encoded.append(vocab.get(word, vocab["<unk>"]))

        encoded.append(vocab["<end>"])

        numerical_captions[image_id].append(encoded)



#### 4.5 Data spliting

In [46]:
# 
#  DATA SPLITTING (ADD HERE)


image_ids = list(numerical_captions.keys())
random.shuffle(image_ids)

split_ratio = 0.8  # 80% train / 20% validation
split_index = int(len(image_ids) * split_ratio)

train_ids = image_ids[:split_index]
val_ids = image_ids[split_index:]

print("Training Images:", len(train_ids))
print("Validation Images:", len(val_ids))

Training Images: 6472
Validation Images: 1619


#### 4.5 save pkl file

In [28]:
# Save Pickle File

with open(NUMERICAL_CAPTIONS_FILE, "wb") as f:
    pickle.dump(numerical_captions, f)

print("Numerical captions saved successfully!")

with open(VOCAB_FILE, "wb") as f:
    pickle.dump(vocab, f)

print("Captions saved to:", OUTPUT_FILE)


Numerical captions saved successfully!
Captions saved to: H:\4. Projects_envs\8. Image caption CNN+LSTM\Images_features.pkl


In [29]:
with open(CAPTION_FILE, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Total lines in captions file:", len(lines))
print("First line:", lines[0])

Total lines in captions file: 40456
First line: image,caption



## 5. Checking data

In [34]:
with open(OUTPUT_FILE, "rb") as f:
    features = pickle.load(f)

with open(NUMERICAL_CAPTIONS_FILE, "rb") as f:
    captions = pickle.load(f)

with open(VOCAB_FILE, "rb") as f:
    vocab = pickle.load(f)

idx2word = {idx: word for word, idx in vocab.items()}

image_id = random.choice(list(features.keys()))

print("\nSelected Image:", image_id)
print("Feature shape:", features[image_id].shape)

caption_encoded = captions[image_id][0]
print("Encoded Caption:", caption_encoded)

decoded_caption = " ".join([idx2word[idx] for idx in caption_encoded])
print("Decoded Caption:", decoded_caption)




Selected Image: 3417662443_2eaea88977.jpg
Feature shape: (2048,)
Encoded Caption: [1, 4, 1243, 74, 97, 4, 3, 3, 916, 594, 3, 35, 3, 13, 1969, 2]
Decoded Caption: <start> a bearded man wears a <unk> <unk> hooded costume <unk> with <unk> of spiderman <end>


## 6. Model creation pytorch

In [ ]:
# LOAD DATA FOR CHECKING

features = features_dict
captions = numerical_captions

image_id = random.choice(list(features.keys()))

print("Selected Image:", image_id)
print("Feature shape:", features[image_id].shape)

caption_encoded = captions[image_id][0]
print("Encoded Caption:", caption_encoded)

decoded_caption = " ".join([idx2word[idx] for idx in caption_encoded])
print("Decoded Caption:", decoded_caption)


Selected Image: 1118557877_736f339752.jpg
Feature shape: (2048,)
Encoded Caption: [1, 4, 22, 140, 132, 6, 24, 250, 2]
Decoded Caption: <start> a little boy stands in the surf <end>


In [ ]:
#MODEL CREATION ( TWO-BRANCH LOGIC)

EMBED_SIZE = 256
HIDDEN_SIZE = 256

# Image branch
feature_fc = nn.Linear(2048, EMBED_SIZE).to(device)
feature_dropout = nn.Dropout(0.4)

# Caption branch
embedding = nn.Embedding(vocab_size, EMBED_SIZE).to(device)
caption_dropout = nn.Dropout(0.4)
lstm = nn.LSTM(EMBED_SIZE, HIDDEN_SIZE, batch_first=True).to(device)

# Decoder
decoder_fc1 = nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE).to(device)
decoder_fc2 = nn.Linear(HIDDEN_SIZE, vocab_size).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)

In [44]:
optimizer = optim.Adam(
    list(feature_fc.parameters()) +
    list(embedding.parameters()) +
    list(lstm.parameters()) +
    list(decoder_fc1.parameters()) +
    list(decoder_fc2.parameters()),
    lr=0.001 )


In [48]:
# IMAGE BRANCH
img = torch.relu(feature_fc(feature))
img = img.unsqueeze(0)  # (1, batch, embed)

# CAPTION INPUT
caption_input = caption[:, :-1]
targets = caption[:, 1:]

emb = embedding(caption_input)

# Initialize hidden state with image features
h0 = img
c0 = torch.zeros_like(h0)

outputs, _ = lstm(emb, (h0, c0))

outputs = decoder_fc2(outputs)

# Reshape for loss
outputs = outputs.reshape(-1, vocab_size)
targets = targets.reshape(-1)

loss = criterion(outputs, targets)

In [49]:
from math import ceil

# SET TRAINING PARAMETERS
# 

epochs = 50
batch_size = 32   #  CHANGE BATCH SIZE HERE

steps_per_epoch = ceil(len(train_ids) / batch_size)
validation_steps = ceil(len(val_ids) / batch_size)

print("Steps per epoch:", steps_per_epoch)
print("Validation steps:", validation_steps)

# TRAINING STARTS
# 

for epoch in range(epochs):

    print(f"\nEpoch {epoch+1}/{epochs}")

    # shuffle training data every epoch
    random.shuffle(train_ids)

    # ================= TRAINING =================
    total_train_loss = 0
    model_step = 0

    for step in range(steps_per_epoch):

        batch_ids = train_ids[step*batch_size : (step+1)*batch_size]

        batch_features = []
        batch_captions = []

        for image_id in batch_ids:

            if image_id not in features:
                continue

            feature_np = features[image_id]
            caption_list = captions[image_id]
            caption_encoded = random.choice(caption_list)

            batch_features.append(feature_np)
            batch_captions.append(caption_encoded)

        if len(batch_features) == 0:
            continue

        # Convert features to tensor
        feature = torch.tensor(batch_features, dtype=torch.float32).to(device)

        # Pad captions
        max_len = max(len(cap) for cap in batch_captions)
        padded_captions = []

        for cap in batch_captions:
            padded = cap + [0]*(max_len - len(cap))
            padded_captions.append(padded)

        caption = torch.tensor(padded_captions, dtype=torch.long).to(device)

        # -------- IMAGE BRANCH --------
        img = feature_dropout(feature)
        img = torch.relu(feature_fc(img))

        # -------- CAPTION BRANCH --------
        caption_input = caption[:, :-1]
        targets = caption[:, 1:]

        emb = embedding(caption_input)
        emb = caption_dropout(emb)

        _, (hidden, _) = lstm(emb)
        text = hidden[-1]

        # -------- MERGE --------
        merged = img + text
        x = torch.relu(decoder_fc1(merged))
        output = decoder_fc2(x)

        loss = criterion(output, targets[:, -1])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        model_step += 1

    print("Train Loss:", total_train_loss / model_step)

    # ================= VALIDATION =================
    total_val_loss = 0
    val_step = 0

    with torch.no_grad():

        for step in range(validation_steps):

            batch_ids = val_ids[step*batch_size : (step+1)*batch_size]

            batch_features = []
            batch_captions = []

            for image_id in batch_ids:

                if image_id not in features:
                    continue

                feature_np = features[image_id]
                caption_list = captions[image_id]
                caption_encoded = random.choice(caption_list)

                batch_features.append(feature_np)
                batch_captions.append(caption_encoded)

            if len(batch_features) == 0:
                continue

            feature = torch.tensor(batch_features, dtype=torch.float32).to(device)

            max_len = max(len(cap) for cap in batch_captions)
            padded_captions = []

            for cap in batch_captions:
                padded = cap + [0]*(max_len - len(cap))
                padded_captions.append(padded)

            caption = torch.tensor(padded_captions, dtype=torch.long).to(device)

            img = torch.relu(feature_fc(feature))

            caption_input = caption[:, :-1]
            targets = caption[:, 1:]

            emb = embedding(caption_input)
            _, (hidden, _) = lstm(emb)
            text = hidden[-1]

            merged = img + text
            x = torch.relu(decoder_fc1(merged))
            output = decoder_fc2(x)

            loss = criterion(output, targets[:, -1])

            total_val_loss += loss.item()
            val_step += 1

    print("Validation Loss:", total_val_loss / val_step)

Steps per epoch: 203
Validation steps: 51

Epoch 1/50
Train Loss: 0.0
Validation Loss: 0.0

Epoch 2/50
Train Loss: 0.0
Validation Loss: 0.0

Epoch 3/50
Train Loss: 0.0
Validation Loss: 0.0

Epoch 4/50
Train Loss: 0.0
Validation Loss: 0.0

Epoch 5/50


KeyboardInterrupt: 

In [ ]:
# 
# CAPTION GENERATION
# 

def generate_caption(image_path, max_length=20):

    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        feature = resnet(image)
        img = torch.relu(feature_fc(feature))

    input_word = torch.tensor([[vocab["<start>"]]]).to(device)
    hidden = None
    result = []

    for _ in range(max_length):

        emb = embedding(input_word)
        output, hidden = lstm(emb, hidden)

        text = hidden[0][-1]
        merged = img + text

        x = torch.relu(decoder_fc1(merged))
        output_word = decoder_fc2(x)

        predicted = output_word.argmax(1)
        word = idx2word[predicted.item()]

        if word == "<end>":
            break

        result.append(word)
        input_word = predicted.unsqueeze(0)

    return " ".join(result)


# TEST GENERATION
test_image = random.choice(train_ids)
image_path = os.path.join(IMAGE_FOLDER, test_image)

print("Generated Caption:")
print(generate_caption(image_path))

In [ ]:
torch.save({
    "feature_fc": feature_fc.state_dict(),
    "embedding": embedding.state_dict(),
    "lstm": lstm.state_dict(),
    "decoder_fc1": decoder_fc1.state_dict(),
    "decoder_fc2": decoder_fc2.state_dict()
}, "image_caption_model.pth")

print("Model saved successfully!")

## 7. Model creation with tf

In [51]:
import tensorflow as tf

print("TensorFlow Version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print("GPU Device:", tf.config.list_physical_devices('GPU'))

ModuleNotFoundError: No module named 'tensorflow'